In [1]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Input, concatenate, Reshape, Conv1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf



# Hàm Load Dữ liệu

In [2]:
import pandas as pd
import numpy as np
from numpy import savetxt, loadtxt

def load_dataset(label_count):
    # Tải dữ liệu từ file CSV
    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0', axis=1)

    # Các cột nhãn
    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Lấy số lượng hàng dữ liệu là 39645
    data = data.iloc[:39645]

    # Chia tập dữ liệu thành tập huấn luyện và tập kiểm tra
    total_rows = len(data)
    split_point = int(0.8 * total_rows)
    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    features_source_train =  np.loadtxt("DATASET/split_train.csv", delimiter=',')
    features_source_test = np.loadtxt("DATASET/split_test.csv", delimiter=',')
    X_train_source = np.array(features_source_train)
    X_test_source = np.array(features_source_test)
    print("Load Features codeBERT success!!")
    # Lấy các đặc trưng từ tập dữ liệu
    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # Lấy nhãn từ tập dữ liệu
    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")
    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels



In [3]:
X_train_opcode_5, X_test_opcode_5, X_train_source_5, X_test_source_5, y_train_5, y_test_5, labels_dataset_full = load_dataset(5)

Load Features codeBERT success!!
Load Data for 5-Label MultiLabel success!!


In [4]:
X_train_opcode_4, X_test_opcode_4, X_train_source_4, X_test_source_4, y_train_4, y_test_4, labels_dataset_4 = load_dataset(4)

Load Features codeBERT success!!
Load Data for 4-Label MultiLabel success!!


In [ ]:
# ===== PyTorch =====
import torch
import torch.nn as nn
import torch.nn.functional as F

# ===== PyG =====
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from torch.nn import BatchNorm1d, Linear

# ===== TF / Keras =====
from tensorflow.keras.models import load_model, Model

# ===== Utils =====
import pickle
import numpy as np

In [5]:
# Load GNN
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np

# Load data
input_file = 'DATASET/gnn_input_binary_labels.pkl'

with open(input_file, 'rb') as f:
    data = pickle.load(f)

loaded_feature_matrices = data['features']
adj_matrices = data['adj_matrices']
# loaded_all_labels = data['labels']
# Define the target dimension
target_dim = 6000

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim))
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features

# Function to create PyTorch Geometric Data object from feature matrices and labels
def create_pyg_data(features, adj_matrix, labels, target_dim):
    num_nodes = features.shape[0]
    edge_index = torch.tensor(np.array(adj_matrix), dtype=torch.long).nonzero(as_tuple=False).t().contiguous()
    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(labels, dtype=torch.float)  # Ensure labels are float for BCEWithLogitsLoss
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

data_list = [create_pyg_data(features, adj_matrix, labels, target_dim) for features, adj_matrix, labels in zip(loaded_feature_matrices, adj_matrices, labels_dataset_full)]

# Create DataLoader
train_loader = DataLoader(data_list[:int(len(data_list)*0.8)], batch_size=32, shuffle=True)  # Adjust batch size as needed
test_loader = DataLoader(data_list[int(len(data_list)*0.8):], batch_size=32, shuffle=False)  # Adjust batch size as needed


In [ ]:
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
        self.bn1 = BatchNorm1d(hidden_channels * heads)

        self.conv2 = GATConv(
            hidden_channels * heads,
            hidden_channels,
            heads=1,
            concat=False,
            dropout=0.3
        )
        self.bn2 = BatchNorm1d(hidden_channels)
        self.lin = Linear(hidden_channels, out_channels)

    def forward_features(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))

        x = global_mean_pool(x, batch)
        return x  # (B, 64)

    def forward(self, data):
        x = self.forward_features(data)
        return torch.sigmoid(self.lin(x))


gat_model = GAT(6000, 64, 8)
gat_model.load_state_dict(torch.load("Unimodal/GAT_30E.pth"))
gat_model.eval()

gat_feats = []
with torch.no_grad():
    for graph_data in train_loader:
        gat_feats.append(gat_model.forward_features(graph_data))

gat_feat = torch.cat(gat_feats, dim=0)  # (B, 64)

C:\Users\Admin\AppData\Local\Temp\ipykernel_1764\214182964.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gnn_model.load_state_dict(torch.load(model_path))


In [ ]:
# ===== BiLSTM =====
bilstm_model = load_model("Unimodal/BiLSTM.h5")
bilstm_feat_model = Model(
    bilstm_model.input,
    bilstm_model.layers[-2].output
)

bilstm_feat = torch.from_numpy(
    bilstm_feat_model(seq_input).numpy()
).float()  # (B, 128)


# ===== CodeBERT =====
codebert_model = load_model("Unimodal/CodeBERT.h5")
codebert_feat_model = Model(
    codebert_model.input,
    codebert_model.layers[-2].output
)

codebert_feat = torch.from_numpy(
    codebert_feat_model([input_ids, attention_mask]).numpy()
).float()  # (B, 768)

# Mô hình khởi tạo

In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten, Reshape, concatenate
from tensorflow.keras.models import load_model

def create_model_with_labels(initial_label_count, total_label_count):
    # Load Bert, Bi-LSTM Model
    bert_model = load_model("Unimodal/CodeBERT_30E.h5", compile=False)
    lstm_model = load_model("Unimodal/BiLSTM_Full_2gram_Softmax_30E.h5", compile=False)

    # Set the models to non-trainable
    bert_model.trainable = False
    lstm_model.trainable = False

    # Combine the outputs of both models
    bert_output = bert_model.output
    lstm_output = lstm_model.output
    gnn_output = tf.keras.layers.Input(shape=(8,), name="gnn_output")  # Kích thước đầu ra của GNN

    # Tắt trainable cho GNN
    gnn_output._keras_history[0].trainable = False
    concatenated = concatenate([bert_output, lstm_output, gnn_output], axis=-1)

    # Feature extractor part
    concatenated_reshaped = Reshape((-1, 1))(concatenated)  # Adjust to match Conv1D input
    conv_out = Conv1D(64, 3, activation='relu')(concatenated_reshaped)
    flatten_out = Flatten()(conv_out)
    dense_features = Dense(128, activation='relu', name='dense_multilabel')(flatten_out)
    dense_features = Dropout(0.3, name='drop_multilabel')(dense_features)

    # Output for the initial_label_count task
    output_initial_labels = Dense(initial_label_count, activation='sigmoid', name='output_initial_labels')(dense_features)

    # Create initial model
    model_initial = tf.keras.models.Model(inputs=[bert_model.input, lstm_model.input, gnn_output], outputs=output_initial_labels)

    # Compile the model
    METRICS = [
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]

    model_initial.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

    return model_initial

In [8]:
initial_label_count = 4
model_initial = create_model_with_labels(initial_label_count, total_label_count=8)
model_initial.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 280)]        0           []                               
                                                                                                  
 sourcecode_features (InputLaye  [(None, 768)]       0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 280, 286)     9724000     ['input_1[0][0]']                
                                                                                                  
 codebert_dense_layer_1 (Dense)  (None, 512)         393728      ['sourcecode_features[0][0]']

In [9]:
model_initial.fit([X_train_source_4, X_train_opcode_4, gnn_train_outputs], y_train_4, epochs=10, batch_size=32, validation_data=([X_test_source_4, X_test_opcode_4, gnn_test_outputs], y_test_4))

Epoch 1/10
992/992 [==============================] - 122s 120ms/step - loss: 0.2626 - accuracy: 0.8968 - precision: 0.8846 - recall: 0.8892 - val_loss: 0.1718 - val_accuracy: 0.9378 - val_precision: 0.9229 - val_recall: 0.9197
Epoch 2/10
992/992 [==============================] - 121s 122ms/step - loss: 0.2411 - accuracy: 0.9034 - precision: 0.8956 - recall: 0.8916 - val_loss: 0.1648 - val_accuracy: 0.9379 - val_precision: 0.9257 - val_recall: 0.9166
Epoch 3/10
992/992 [==============================] - 119s 120ms/step - loss: 0.2417 - accuracy: 0.9032 - precision: 0.8967 - recall: 0.8897 - val_loss: 0.1638 - val_accuracy: 0.9383 - val_precision: 0.9293 - val_recall: 0.9137
Epoch 4/10
992/992 [==============================] - 116s 117ms/step - loss: 0.2385 - accuracy: 0.9047 - precision: 0.8993 - recall: 0.8902 - val_loss: 0.1589 - val_accuracy: 0.9389 - val_precision: 0.9294 - val_recall: 0.9151
Epoch 5/10
992/992 [==============================] - 118s 119ms/step - loss: 0.2383 - a

ResourceExhaustedError: Graph execution error:

OOM when allocating tensor with shape[280,32,286] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator cpu
	 [[{{node ReverseV2}}]]
Hint: If you want to see a list of allocated tensors when OOM happens, add report_tensor_allocations_upon_oom to RunOptions for current allocation info. This isn't available when running in Eager mode.

	 [[model/bidirectional/backward_lstm/PartitionedCall]]
Hint: If you want to see a list of allocated tensors when OOM happens, add report_tensor_allocations_upon_oom to RunOptions for current allocation info. This isn't available when running in Eager mode.
 [Op:__inference_train_function_7294]

# Model mở rộng

In [ ]:
def extend_model(initial_model, total_label_count):
    # Freeze all layers in the initial model
    for layer in initial_model.layers:
        layer.trainable = False

    # Get the output of the second last layer of the initial model
    dense_features = initial_model.get_layer('drop_multilabel').output

    # Reshape the output to match the expected input shape of the new layer
    dense_features = Reshape((128,), name ='reshape_extend')(dense_features)

    # Create new output layer for the additional labels
    output_extended_labels = Dense(total_label_count, activation='sigmoid', name='output_extended_labels')(dense_features)

    # Create extended model
    model_extended = tf.keras.models.Model(inputs=initial_model.inputs, outputs=output_extended_labels)

    # Compile the extended model
    METRICS = [
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]

    model_extended.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

    return model_extended

In [ ]:
total_label_count = 5
extended_model = extend_model(model_initial, total_label_count)
extended_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 280)]        0           []                               
                                                                                                  
 sourcecode_features (InputLaye  [(None, 768)]       0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 280, 286)     9724000     ['input_1[0][0]']                
                                                                                                  
 codebert_dense_layer_1 (Dense)  (None, 512)         393728      ['sourcecode_features[0][0]

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)

In [ ]:
extended_model.fit([X_train_source_5, X_train_opcode_5, gnn_train_outputs], y_train_5, epochs=1, batch_size=32, callbacks=[early_stopping])

992/992 [==============================] - 82s 79ms/step - loss: 0.3072 - accuracy: 0.8722 - precision: 0.8632 - recall: 0.8103


In [ ]:
def apply_threshold(y_pred_prob, threshold=0.5):
    y_pred_prob = np.array(y_pred_prob)
    y_pred_thresh = np.zeros_like(y_pred_prob)
    y_pred_thresh[y_pred_prob >= threshold] = 1
    return y_pred_thresh
y_pred_prob = extended_model.predict([X_test_source_5, X_test_opcode_5, gnn_test_outputs])
y_pred = apply_threshold(y_pred_prob, threshold=0.5)

248/248 [==============================] - 21s 83ms/step


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
label_names = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

for i, label in enumerate(label_names[:5]):
    precision = precision_score(y_test_5[:, i], y_pred[:, i])
    recall = recall_score(y_test_5[:, i], y_pred[:, i])
    f1 = f1_score(y_test_5[:, i], y_pred[:, i])
    accuracy = accuracy_score(y_test_5[:, i], y_pred[:, i])
    # Tính confusion matrix cho nhãn i
    tn, fp, fn, tp = confusion_matrix(y_test_5[:, i], y_pred[:, i]).ravel()

    # Tính FPR và FNR
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    print(f"Metrics for {label}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    #print(f"  FPR:       {fpr:.4f}")
    #print(f"  FNR:       {fnr:.4f}")
    print()

Metrics for Other:
  Accuracy:  0.8811
  Precision: 0.8967
  Recall:    0.8925
  F1-Score:  0.8946

Metrics for access_control:
  Accuracy:  0.9572
  Precision: 0.9301
  Recall:    0.5350
  F1-Score:  0.6793

Metrics for arithmetic:
  Accuracy:  0.9499
  Precision: 0.9564
  Recall:    0.9788
  F1-Score:  0.9675

Metrics for denial_service:
  Accuracy:  0.9637
  Precision: 0.8887
  Recall:    0.9030
  F1-Score:  0.8958

Metrics for front_running:
  Accuracy:  0.8666
  Precision: 0.6079
  Recall:    0.2762
  F1-Score:  0.3798



IndexError: index 5 is out of bounds for axis 1 with size 5

# END

# 5 đến 8

In [ ]:
X_train_opcode_5, X_test_opcode_5, X_train_source_5, X_test_source_5, y_train_5, y_test_5, labels_dataset_5 = load_dataset(5)

Load Features codeBERT success!!
Load Data for 5-Label MultiLabel success!!


In [ ]:
initial_label_count = 5
model_initial = create_model_with_labels(initial_label_count, total_label_count=8)
model_initial.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
model_initial.fit([X_train_source_5, X_train_opcode_5, gnn_train_outputs], y_train_5, epochs=30, batch_size=32, validation_data=([X_test_source_5, X_test_opcode_5, gnn_test_outputs], y_test_5))

Epoch 1/30
992/992 [==============================] - 123s 120ms/step - loss: 0.2622 - accuracy: 0.8964 - precision: 0.8766 - recall: 0.8639 - val_loss: 0.1657 - val_accuracy: 0.9411 - val_precision: 0.9190 - val_recall: 0.9101
Epoch 2/30
992/992 [==============================] - 120s 121ms/step - loss: 0.2389 - accuracy: 0.9055 - precision: 0.8849 - recall: 0.8793 - val_loss: 0.1591 - val_accuracy: 0.9413 - val_precision: 0.9293 - val_recall: 0.8988
Epoch 3/30
992/992 [==============================] - 129s 130ms/step - loss: 0.2380 - accuracy: 0.9050 - precision: 0.8863 - recall: 0.8762 - val_loss: 0.1594 - val_accuracy: 0.9410 - val_precision: 0.9262 - val_recall: 0.9016
Epoch 4/30
992/992 [==============================] - 124s 125ms/step - loss: 0.2371 - accuracy: 0.9058 - precision: 0.8875 - recall: 0.8768 - val_loss: 0.1553 - val_accuracy: 0.9418 - val_precision: 0.9253 - val_recall: 0.9049
Epoch 5/30
992/992 [==============================] - 116s 117ms/step - loss: 0.2352 - a

In [ ]:
total_label_count = 8
extended_model = extend_model(model_initial, 5, total_label_count)
extended_model.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
extended_model.fit([X_train_source_8, X_train_opcode_8, gnn_train_outputs], y_train_8, epochs=30, batch_size=32, callbacks=[early_stopping])

Epoch 1/30
992/992 [==============================] - 118s 116ms/step - loss: 0.3456 - accuracy: 0.8498 - precision: 0.8357 - recall: 0.6841
Epoch 2/30
992/992 [==============================] - 113s 114ms/step - loss: 0.2852 - accuracy: 0.8788 - precision: 0.8686 - recall: 0.7500
Epoch 3/30
992/992 [==============================] - 112s 113ms/step - loss: 0.2791 - accuracy: 0.8816 - precision: 0.8635 - recall: 0.7661
Epoch 4/30
992/992 [==============================] - 111s 112ms/step - loss: 0.2762 - accuracy: 0.8827 - precision: 0.8611 - recall: 0.7731
Epoch 5/30
992/992 [==============================] - 112s 113ms/step - loss: 0.2756 - accuracy: 0.8830 - precision: 0.8586 - recall: 0.7771
Epoch 6/30
992/992 [==============================] - 110s 111ms/step - loss: 0.2754 - accuracy: 0.8835 - precision: 0.8583 - recall: 0.7792
Epoch 7/30
992/992 [==============================] - 109s 110ms/step - loss: 0.2753 - accuracy: 0.8834 - precision: 0.8579 - recall: 0.7795
Epoch 8/30
99

In [ ]:
extended_model.save('KLTN_5to8.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


# 6 to 8

In [ ]:
X_train_opcode_6, X_test_opcode_6, X_train_source_6, X_test_source_6, y_train_6, y_test_6, labels_dataset_6 = load_dataset(6)

Load Features codeBERT success!!
Load Data for 6-Label MultiLabel success!!


In [ ]:
initial_label_count = 6
model_initial = create_model_with_labels(initial_label_count, total_label_count=8)
model_initial.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
model_initial.fit([X_train_source_6, X_train_opcode_6, gnn_train_outputs], y_train_6, epochs=30, batch_size=32, validation_data=([X_test_source_6, X_test_opcode_6, gnn_test_outputs], y_test_6))

Epoch 1/30
992/992 [==============================] - 172s 169ms/step - loss: 0.2744 - accuracy: 0.8922 - precision: 0.8674 - recall: 0.8478 - val_loss: 0.1835 - val_accuracy: 0.9352 - val_precision: 0.9047 - val_recall: 0.9044
Epoch 2/30
992/992 [==============================] - 168s 169ms/step - loss: 0.2537 - accuracy: 0.8998 - precision: 0.8744 - recall: 0.8620 - val_loss: 0.1766 - val_accuracy: 0.9360 - val_precision: 0.9201 - val_recall: 0.8885
Epoch 3/30
992/992 [==============================] - 162s 163ms/step - loss: 0.2518 - accuracy: 0.8997 - precision: 0.8747 - recall: 0.8612 - val_loss: 0.1696 - val_accuracy: 0.9367 - val_precision: 0.9153 - val_recall: 0.8965
Epoch 4/30
992/992 [==============================] - 169s 170ms/step - loss: 0.2475 - accuracy: 0.9009 - precision: 0.8760 - recall: 0.8632 - val_loss: 0.1711 - val_accuracy: 0.9354 - val_precision: 0.9085 - val_recall: 0.9004
Epoch 5/30
992/992 [==============================] - 160s 161ms/step - loss: 0.2485 - a

In [ ]:
total_label_count = 8
extended_model = extend_model(model_initial, 6, total_label_count)
extended_model.summary()

Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
extended_model.fit([X_train_source_8, X_train_opcode_8, gnn_train_outputs], y_train_8, epochs=30, batch_size=32, callbacks=[early_stopping])

Epoch 1/30
992/992 [==============================] - 111s 108ms/step - loss: 0.3306 - accuracy: 0.8590 - precision: 0.8252 - recall: 0.7321
Epoch 2/30
992/992 [==============================] - 106s 107ms/step - loss: 0.2623 - accuracy: 0.8916 - precision: 0.8679 - recall: 0.7962
Epoch 3/30
992/992 [==============================] - 110s 111ms/step - loss: 0.2560 - accuracy: 0.8931 - precision: 0.8661 - recall: 0.8037
Epoch 4/30
992/992 [==============================] - 116s 117ms/step - loss: 0.2544 - accuracy: 0.8939 - precision: 0.8653 - recall: 0.8074
Epoch 5/30
992/992 [==============================] - 116s 117ms/step - loss: 0.2531 - accuracy: 0.8939 - precision: 0.8643 - recall: 0.8088
Epoch 6/30
992/992 [==============================] - 118s 119ms/step - loss: 0.2527 - accuracy: 0.8945 - precision: 0.8643 - recall: 0.8109
Epoch 7/30
992/992 [==============================] - 120s 121ms/step - loss: 0.2525 - accuracy: 0.8945 - precision: 0.8638 - recall: 0.8115
Epoch 8/30
99

In [ ]:
extended_model.save('KLTN_6to8.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


# 7 to 8

In [ ]:
X_train_opcode_7, X_test_opcode_7, X_train_source_7, X_test_source_7, y_train_7, y_test_7, labels_dataset_7 = load_dataset(7)

Load Features codeBERT success!!
Load Data for 7-Label MultiLabel success!!


In [ ]:
initial_label_count = 7
model_initial = create_model_with_labels(initial_label_count, total_label_count=8)
model_initial.summary()

Model: "model_6"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
model_initial.fit([X_train_source_7, X_train_opcode_7, gnn_train_outputs], y_train_7, epochs=30, batch_size=32, validation_data=([X_test_source_7, X_test_opcode_7, gnn_test_outputs], y_test_7))

Epoch 1/30
992/992 [==============================] - 148s 146ms/step - loss: 0.2755 - accuracy: 0.8924 - precision: 0.8586 - recall: 0.8200 - val_loss: 0.1788 - val_accuracy: 0.9384 - val_precision: 0.9150 - val_recall: 0.8762
Epoch 2/30
992/992 [==============================] - 146s 147ms/step - loss: 0.2495 - accuracy: 0.9019 - precision: 0.8672 - recall: 0.8418 - val_loss: 0.1721 - val_accuracy: 0.9387 - val_precision: 0.9059 - val_recall: 0.8879
Epoch 3/30
992/992 [==============================] - 146s 147ms/step - loss: 0.2463 - accuracy: 0.9030 - precision: 0.8690 - recall: 0.8430 - val_loss: 0.1717 - val_accuracy: 0.9370 - val_precision: 0.9112 - val_recall: 0.8756
Epoch 4/30
992/992 [==============================] - 144s 145ms/step - loss: 0.2448 - accuracy: 0.9036 - precision: 0.8703 - recall: 0.8436 - val_loss: 0.1681 - val_accuracy: 0.9393 - val_precision: 0.9065 - val_recall: 0.8895
Epoch 5/30
992/992 [==============================] - 142s 143ms/step - loss: 0.2435 - a

In [ ]:
total_label_count = 8
extended_model = extend_model(model_initial, 7, total_label_count)
extended_model.summary()

Model: "model_7"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 280)]                0         []                            
                                                                                                  
 sourcecode_features (Input  [(None, 768)]                0         []                            
 Layer)                                                                                           
                                                                                                  
 embedding (Embedding)       (None, 280, 286)             9724000   ['input_1[0][0]']             
                                                                                                  
 codebert_dense_layer_1 (De  (None, 512)                  393728    ['sourcecode_features[0]

In [ ]:
extended_model.fit([X_train_source_8, X_train_opcode_8, gnn_train_outputs], y_train_8, epochs=30, batch_size=32, callbacks=[early_stopping])

Epoch 1/30
992/992 [==============================] - 119s 117ms/step - loss: 0.3319 - accuracy: 0.8552 - precision: 0.8131 - recall: 0.7348
Epoch 2/30
992/992 [==============================] - 117s 118ms/step - loss: 0.2620 - accuracy: 0.8908 - precision: 0.8628 - recall: 0.7998
Epoch 3/30
992/992 [==============================] - 117s 118ms/step - loss: 0.2578 - accuracy: 0.8923 - precision: 0.8620 - recall: 0.8061
Epoch 4/30
992/992 [==============================] - 117s 118ms/step - loss: 0.2561 - accuracy: 0.8931 - precision: 0.8616 - recall: 0.8094
Epoch 5/30
992/992 [==============================] - 114s 115ms/step - loss: 0.2563 - accuracy: 0.8929 - precision: 0.8611 - recall: 0.8094
Epoch 6/30
992/992 [==============================] - 114s 115ms/step - loss: 0.2554 - accuracy: 0.8934 - precision: 0.8618 - recall: 0.8105
Epoch 7/30
992/992 [==============================] - 116s 117ms/step - loss: 0.2558 - accuracy: 0.8928 - precision: 0.8615 - recall: 0.8086
Epoch 8/30
99

In [ ]:
extended_model.save('Transfer_learning_7to8.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


# Kiểm thử